# California species head for BioCLIP 2

Trains a linear classifier on frozen BioCLIP 2 image embeddings of the crops made by
`training/prepare.py`, then writes `species_head.pt`. Copy that file to `models/` and the
app uses it automatically (zero-shot remains the fallback).

**Run on Kaggle with a GPU:** upload `training/data/crops/` and `training/data/splits.csv`
as a dataset, attach it, and set `DATA` below. CPU works too, just slowly.

Ship rule: only replace zero-shot if this beats it on top-1, per-species recall, **and**
lookalike confusion on the held-out test split (cell 6).

In [ ]:
!pip -q install open_clip_torch

In [ ]:
import csv, json, math, random
from pathlib import Path
import numpy as np, torch, open_clip
from PIL import Image

DATA = Path("/kaggle/input/fishid-crops")        # folder with crops/ and splits.csv
OUT = Path("/kaggle/working")
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); random.seed(0)

rows = list(csv.DictReader(open(DATA / "splits.csv")))
ids = sorted({r["species_id"] for r in rows})
index = {s: i for i, s in enumerate(ids)}
print(len(rows), "crops,", len(ids), "species")

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip-2")
model = model.to(device).eval()

@torch.no_grad()
def embed(paths, bs=64):
    out = []
    for i in range(0, len(paths), bs):
        x = torch.stack([preprocess(Image.open(DATA / p).convert("RGB")) for p in paths[i:i + bs]]).to(device)
        f = model.encode_image(x)
        out.append((f / f.norm(dim=-1, keepdim=True)).float().cpu())
    return torch.cat(out)

split = {s: [r for r in rows if r["split"] == s] for s in ("train", "val", "test")}
X = {s: embed([r["path"] for r in split[s]]) for s in split}
Y = {s: torch.tensor([index[r["species_id"]] for r in split[s]]) for s in split}
{s: tuple(X[s].shape) for s in X}

In [ ]:
# Linear probe with class-balanced loss and early stopping on validation accuracy.
D, C = X["train"].shape[1], len(ids)
head = torch.nn.Linear(D, C)
counts = torch.bincount(Y["train"], minlength=C).float().clamp(min=1)
loss_fn = torch.nn.CrossEntropyLoss(weight=(counts.sum() / counts) ** 0.5)
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
best, best_state, patience = 0, None, 0
for epoch in range(300):
    head.train()
    perm = torch.randperm(len(Y["train"]))
    for i in range(0, len(perm), 256):
        b = perm[i:i + 256]
        opt.zero_grad()
        loss_fn(head(X["train"][b] * 10), Y["train"][b]).backward()
        opt.step()
    head.eval()
    with torch.no_grad():
        acc = (head(X["val"] * 10).argmax(1) == Y["val"]).float().mean().item()
    if acc > best:
        best, best_state, patience = acc, {k: v.clone() for k, v in head.state_dict().items()}, 0
    else:
        patience += 1
        if patience > 25:
            break
head.load_state_dict(best_state)
print(f"best val top-1 {best:.3f} at epoch {epoch - patience}")

In [ ]:
# Temperature scaling on validation so probabilities are calibrated (the app's
# 'confident' threshold depends on it).
with torch.no_grad():
    logits_val = head(X["val"] * 10)
T = torch.ones(1, requires_grad=True)
opt_t = torch.optim.LBFGS([T], lr=0.1, max_iter=100)
def closure():
    opt_t.zero_grad()
    l = torch.nn.functional.cross_entropy(logits_val / T, Y["val"])
    l.backward()
    return l
opt_t.step(closure)
temperature = float(T.item())
print("temperature", temperature)

In [ ]:
# Held-out test: top-1, top-3, per-species recall, and the lookalike pairs that matter for regulations.
with torch.no_grad():
    probs = torch.softmax(head(X["test"] * 10) / temperature, dim=1)
top = probs.topk(3, dim=1).indices
y = Y["test"]
print(f"test top-1 {(top[:, 0] == y).float().mean():.3f}   top-3 {(top == y[:, None]).any(1).float().mean():.3f}")
recall = {}
for s, i in index.items():
    m = y == i
    if m.any():
        recall[s] = (top[m, 0] == i).float().mean().item()
worst = sorted(recall.items(), key=lambda t: t[1])[:15]
print("lowest recall:", worst)
pairs = [("vermilion_rockfish", "yelloweye_rockfish"), ("vermilion_rockfish", "canary_rockfish"),
         ("copper_rockfish", "quillback_rockfish"), ("chinook_salmon", "coho_salmon"),
         ("white_sturgeon", "green_sturgeon"), ("kelp_bass", "barred_sand_bass")]
for a, b in pairs:
    if a in index and b in index:
        m = y == index[a]
        if m.any():
            print(f"{a} predicted as {b}: {(top[m, 0] == index[b]).float().mean():.1%}")

In [ ]:
torch.save({"ids": ids, "weight": head.weight.detach() * 10, "bias": head.bias.detach(),
            "temperature": temperature}, OUT / "species_head.pt")
print("wrote", OUT / "species_head.pt", "-> copy to models/species_head.pt")

### If the linear probe misses the targets

Unfreeze the last 2–4 transformer blocks of the visual encoder and fine-tune with a small
learning rate (1e-5) and strong augmentation (random resized crop, color jitter, horizontal
flip). Export the same `species_head.pt` format plus the fine-tuned visual weights, and load
them in `fishid/species/finetuned.py`.